**Caso Estudio - Primer Método Integración**

El primer método de integración se conoce como el método de muestreo (muestreo por clusterización - Montecarlo). Este método esta determinado por el índice de confiabilidad. De manera general la confiabilidad a norma internacional esta marcado por una confiabilidad del 99.9%, lo que requerira como minimo tener un total de 1000 datos.

0. Cargamos las librerias de trabajo

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns  #Las gráficas estadísticas
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
path ="/content/drive/MyDrive/Integracion de datos/1. RiesgoOperacional_EVERGREEEN.xlsx"

1. Cargamos los datos de trabajo

In [ ]:
XDB=pd.read_excel(path,sheet_name=0)
#display(XDB)
Freq=XDB.iloc[:,3]; Sev=XDB.iloc[:,4]
LDA=Freq*Sev

plt.figure()
sns.kdeplot(LDA)
plt.grid()
plt.show()

2. Se procede con la creación de los concentradores de información

In [ ]:
import random
random.seed(42)
XC=np.array(random.choices(LDA,k=5))
XC=np.sort(XC)
print("Los concentradores de información son:",XC)

3. Se clasifican los datos de acuerdo con los concentradores de información

In [ ]:
import random
random.seed(42)
XC=np.array(random.choices(LDA,k=5))
XC=np.sort(XC)

box0=[];box1=[];box2=[];box3=[];box4=[]
Boxes=[box0,box1,box2,box3,box4]

for k in range(len(LDA)):
  d=np.abs(XC-LDA.iloc[k,])/XC #la distancia de un dato a un concentrador
  nc=np.argmin(d)  #Número de la caja en donde voy a introducir el dato
  #No olvidar actualizar el centoride de la caja
  XC[nc] = (XC[nc]+LDA.iloc[k,])/2
  Boxes[nc].append(LDA.iloc[k,])

#Determinamos el porcentaje de datos por Caja
pm=[]  #Vector de porcentajes
for j in range(5):
  pd=len(Boxes[j])/len(LDA)
  pm.append(pd)

pm=(np.array(pm)*1000).astype(int)
print("El porcentaje de los datos es:",pm)

4. Sacamos daticos por caja y creamos una LDA2

In [ ]:
LDA2=[]  #Aquí van a quedar los 1000 datos

for j in range(5):
  LDA2.extend(random.choices(Boxes[j],k=pm[j]))

LDA2=np.array(LDA2)

#Se comprueban las métricas estadísticas
uo=np.mean(LDA)
print("la media de los datos observados es:",uo)
ue=np.mean(LDA2)
print("la media de los datos externos es:",ue)
du=np.abs((uo-ue)/uo)
print("El error relativo porcentual es:",du*100)

plt.figure()
sns.kdeplot(LDA,color='red',label='Datos Observados')
sns.kdeplot(LDA2,color='blue',label='Datos Externos')
plt.grid()
plt.show()
#

**Análisis de Resultados**
De acuerdo con el método de integración de datos por muestreo aleatorio y concentradores de información, la discrepancia que se obtuvo para la media de las distribuciones (Datos Observados:LDA, Datos Externos:LDA2) estuvo por debajo del 5% en promedio, lo que indica que el 95% de los datos aproximadamente fueron correctamente seleccionados. Este proceso de muestreo se hizo con el fin de lograr una confiabilidad del 99,9% en la estimación de las pérdidas por dejar contenedores en tierra. Es importante mencionar que las medias de los datos observadoes y los externos fueron bla, bla, bla, las desviaciones fueron bla, bla, bla, los coeficientes asimetría fueron tales y tales, y la curtosis fueron tales y tales (es que fueron muy similares).

**Intefracion datos cosco (DATSO EXTERNOS)**

En esta parte la naciera ecergreen quiere ingegrar los datos logisticos de la operacion de cosco shipping, para lo cua lutilizara el modelo k-medoids

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ruta del archivo
path2 = "/content/drive/MyDrive/Integracion de datos/3. Datos COSCO Shipping.xlsx"

# Leer archivo Excel
XDB2 = pd.read_excel(path2, sheet_name=0)

# Mostrar base de datos
display(XDB2)

# Tomar frecuencia y severidad desde el DataFrame
Freqe = pd.to_numeric(XDB2.iloc[:, 3], errors='coerce')
Seve = pd.to_numeric(XDB2.iloc[:, 4], errors='coerce')

# Calcular LDA
LDAe = Freqe * Seve

# Eliminar valores vacíos o no numéricos
LDAe = LDAe.dropna()

# Media
print("La media de los datos es:", np.mean(LDAe))

# Asimetría
print(
    "El coeficiente de asimetría (skewness) es:",
    LDAe.skew()
)

# Curtosis
print(
    "El coeficiente de curtosis es:",
    LDAe.kurt()
)

# Gráfica
plt.figure(figsize=(8, 5))

sns.kdeplot(LDAe)

plt.title("Distribución agregada de pérdidas - LDA")
plt.xlabel("Pérdida")
plt.ylabel("Densidad")
plt.grid()

plt.show()

2. se procede con la integracion de los datos externos

In [ ]:
LDAe=np.array(LDAe)

In [ ]:
# Asegurar que XC sea un arreglo numérico
XC = np.asarray(XC, dtype=float)

# Asegurar que LDAe sea un arreglo numérico
LDAe = np.asarray(LDAe, dtype=float)

# Tomar el primer dato de LDAe
dato = LDAe[0]

# Calcular VP
VP = np.exp(-0.5 * ((XC - dato) / XC)**2)

print("VP:", VP)
print("los valores de pertenencia a los clusters internos son;",VP)

if np.argmax(VP)>0.90:
  print("Se integra el dato externo")
  nc= np.argmax(VP)
  Boxes[nc].append(LDAe[0,])
  XC[nc,]=(Xc[nc,]+LDAe[0,])/2

In [ ]:
# Contador de datos integrados
m1 = 0

for k in range(len(LDAe)):

    # Calcular valores de pertenencia
    VP = np.exp(-0.5 * ((XC - LDAe[k]) / XC)**2)

    # Verificar si el mayor valor de pertenencia supera 0.90
    if np.max(VP) > 0.90:

        # Contar el dato como integrado
        m1 = m1 + 1

        # Identificar la caja con mayor pertenencia
        nc = np.argmax(VP)

        # Agregar el dato actual a la caja
        Boxes[nc].append(LDAe[k])

        # Actualizar el centroide
        XC[nc] = (XC[nc] + LDAe[k]) / 2

# Mostrar total de datos integrados
print("Los datos integrados fueron:", m1)

3. ahora procedemos con el muestreo de las cajas


In [ ]:
import scipy.stats as stats
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns


# --------------------------------------------------
# 1. CREAR LA DISTRIBUCIÓN EXTERNA MUESTREADA
# --------------------------------------------------

LDA2e = []

for j in range(5):
    LDA2e.extend(random.choices(Boxes[j], k=pm[j]))

LDA2e = np.array(LDA2e)


# --------------------------------------------------
# 2. GRÁFICA DE LAS TRES DISTRIBUCIONES
# --------------------------------------------------

plt.figure()

sns.kdeplot(LDA, color='red', label='Datos Observados')
sns.kdeplot(LDA2, color='orange', label='Datos Muestreados')
sns.kdeplot(LDA2e, color='blue', label='Datos Muestreados con los Externos')

plt.title("Comparación de distribuciones")
plt.xlabel("Pérdida")
plt.ylabel("Densidad")

plt.legend()
plt.grid()
plt.show()


# --------------------------------------------------
# 3. MÉTRICAS DE LA DISTRIBUCIÓN ORIGINAL
# --------------------------------------------------

p1, p2, p3, p4, p5 = caracterizacion(LDA)

print("DISTRIBUCIÓN ORIGINAL")
print("La media de la distribución original es:", p1)
print("La varianza de la distribución original es:", p2)
print("El coeficiente de asimetría de la distribución original es:", p3)
print("La curtosis de la distribución original es:", p4)
print("La desviación estándar de la distribución original es:", p5)


# --------------------------------------------------
# 4. MÉTRICAS DE LA DISTRIBUCIÓN MUESTREADA
# --------------------------------------------------

p1, p2, p3, p4, p5 = caracterizacion(LDA2)

print("\nDISTRIBUCIÓN MUESTREADA")
print("La media de la distribución muestreada es:", p1)
print("La varianza de la distribución muestreada es:", p2)
print("El coeficiente de asimetría de la distribución muestreada es:", p3)
print("La curtosis de la distribución muestreada es:", p4)
print("La desviación estándar de la distribución muestreada es:", p5)


# --------------------------------------------------
# 5. MÉTRICAS DE LA DISTRIBUCIÓN INTEGRADA
# --------------------------------------------------

p1, p2, p3, p4, p5 = caracterizacion(LDA2e)

print("\nDISTRIBUCIÓN INTEGRADA CON DATOS EXTERNOS")
print("La media de la distribución integrada es:", p1)
print("La varianza de la distribución integrada es:", p2)
print("El coeficiente de asimetría de la distribución integrada es:", p3)
print("La curtosis de la distribución integrada es:", p4)
print("La desviación estándar de la distribución integrada es:", p5)

4. se inegra el metodo para la caracterizacion de vairables aleatrias

In [ ]:
def caracterizacion(LDA):

  #Se procede con la creación de la distribución
  plt.figure(figsize=(10,5))
  sns.histplot(LDA,bins=10,kde=True)
  plt.xlabel("Pérdida (KUSD)")
  plt.grid()
  plt.show()

  #Se procede con la caracterización de cada una de las variables
  np.set_printoptions(suppress=True)
  NI=10    #Indica el número de clusters
  counts,bin_edges=np.histogram(LDA,bins=NI)
  print("El número de datos por intervalo es:")
  print(counts)
  print("Los intervalos inferiores:")
  print(bin_edges[:-1])
  print("Los intervalos superiores:")
  print(bin_edges[1:])
  XC=(bin_edges[:-1]+bin_edges[1:])/2

  #Se configura la tabla de los datos
  Tabla=np.column_stack((bin_edges[:-1],bin_edges[1:],XC,counts))
  df=pd.DataFrame(Tabla,columns=['LI','LS','XC','ND'])
  df.head(10)

  #Se procede con la estimación de la media
  fr=counts/np.sum(counts)
  u=np.sum(XC*fr)
  sigma2=np.sum(fr*(XC-u)**2)
  sigma=np.sqrt(sigma2)
  Cas=np.sum(fr*(XC-u)**3)/sigma**3
  Kur=(np.sum(fr*(XC-u)**4)/sigma**4)-3

  return u,sigma,Cas,Kur,df